# 03. 모델 실험

기본 행동 변수로 Logistic Regression, Random Forest, XGBoost를 비교합니다.
이후 클래스 불균형 보정과 추가 피처의 필요성을 확인합니다.

**평가 지표:** Accuracy, Precision, Recall, F1-score, ROC-AUC

In [ ]:
from pathlib import Path

def find_project_root() -> Path:
    current = Path.cwd().resolve()
    if current.name == "notebooks":
        return current.parent
    if (current / "notebooks").exists():
        return current
    return current.parent

PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SAMPLE_DIR = PROJECT_ROOT / "data" / "sample"
MODEL_DIR = PROJECT_ROOT / "models"

for directory in [INTERIM_DIR, PROCESSED_DIR, SAMPLE_DIR, MODEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score,
)
from xgboost import XGBClassifier

master = pd.read_csv(PROCESSED_DIR / "master_model_ready.csv", parse_dates=["trial_date"])
print("modeling sample:", master.shape)

In [ ]:
BASE_FEATURES = [
    "avg_stay_hour",
    "avg_daily_enter",
    "visit_days",
    "first_visit_delay",
    "consecutive_group_2일",
    "consecutive_group_3일",
    "first_visit_hour",
    "n_sites_visited",
    "area_pyeong",
    "is_post_covid",
]

X = master[BASE_FEATURES].copy()
y = master["is_payment"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

medians = X_train.median(numeric_only=True)
X_train_fill = X_train.fillna(medians)
X_test_fill = X_test.fillna(medians)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_fill)
X_test_scaled = scaler.transform(X_test_fill)

In [ ]:
neg = int((y_train == 0).sum())
pos = int((y_train == 1).sum())
scale_pos_weight = neg / pos

models = {
    "Logistic Regression": (
        LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
        X_train_scaled,
        X_test_scaled,
    ),
    "Random Forest": (
        RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        ),
        X_train_fill,
        X_test_fill,
    ),
    "XGBoost": (
        XGBClassifier(
            n_estimators=300,
            max_depth=3,
            learning_rate=0.05,
            scale_pos_weight=scale_pos_weight,
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1,
        ),
        X_train_fill,
        X_test_fill,
    ),
}

rows = []

for name, (model, train_x, test_x) in models.items():
    model.fit(train_x, y_train)
    pred = model.predict(test_x)
    proba = model.predict_proba(test_x)[:, 1]

    rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred),
        "recall": recall_score(y_test, pred),
        "f1": f1_score(y_test, pred),
        "roc_auc": roc_auc_score(y_test, proba),
    })

result = pd.DataFrame(rows).set_index("model").round(4)
result

## 실험 결론

단순 정확도보다 **결제 가능 고객을 놓치지 않는 Recall**을 운영상 중요 지표로 봅니다.
XGBoost는 비선형 관계와 변수 간 상호작용을 학습할 수 있어 최종 튜닝 대상으로 선정합니다.

다음 노트북에서 기본 10개 행동 변수에 상호작용·비율·시간대·로그 변환 변수를 추가해
총 **42개 피처**로 확장합니다.

In [ ]:
master.to_csv(
    PROCESSED_DIR / "model_experiment_data.csv",
    index=False,
    encoding="utf-8-sig",
)
print("저장 완료: data/processed/model_experiment_data.csv")